# 00 - Environment setup
**ML Project 2 - E-Commerce Delivery Risk Management (Olist)** - Ajay Uppu

Run this notebook first, top to bottom. It
1. checks the Python version and the pinned libraries (installs them if they differ),
2. makes sure the 9 raw Olist CSV files are in `data/raw/` (downloads them from the project S3 bucket if they are missing),
3. verifies the files (row counts and sizes) so every later notebook starts from identical data.

> If cell 3 reports that it installed packages, use **Kernel -> Restart Kernel** and run this notebook again.

In [1]:
# Cell 1 - make the project package importable and define folders
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)
from olist_risk import config
config.ensure_dirs()
print("PROJECT_ROOT =", PROJECT_ROOT)
print("outputs ->", config.OUTPUT_DIR)

PROJECT_ROOT = /home/sagemaker-user/ml-project2-v2
outputs -> /home/sagemaker-user/ml-project2-v2/outputs


In [2]:
# Cell 2 - Python and library versions actually used by this kernel
import platform, importlib
print("python", platform.python_version())
pinned = {l.split("==")[0]: l.split("==")[1].strip() for l in open(PROJECT_ROOT / "requirements.txt")
          if "==" in l and not l.startswith("#")}
status = []
for pkg, want in pinned.items():
    mod = {"scikit-learn": "sklearn"}.get(pkg, pkg)
    try:
        have = importlib.import_module(mod).__version__
    except Exception:
        have = None
    status.append({"package": pkg, "required": want, "installed": have, "ok": have == want})
status = pd.DataFrame(status); display(status)

python 3.12.15


,package,required,installed,ok
0,numpy,2.2.6,2.2.6,True
1,pandas,2.2.3,2.2.3,True
2,scipy,1.15.3,1.15.3,True
3,scikit-learn,1.6.1,1.6.1,True
4,lightgbm,4.6.0,4.6.0,True
5,joblib,1.4.2,1.4.2,True
6,matplotlib,3.10.0,3.10.0,True


In [3]:
# Cell 3 - install the pinned versions only if something is missing/different
if not status.ok.all():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")], check=True)
    print("Installed pinned requirements -> restart the kernel and re-run this notebook.")
else:
    print("All pinned packages already installed - nothing to do.")

All pinned packages already installed - nothing to do.


In [4]:
# Cell 4 - make sure the raw CSVs are present (download from S3 if missing)
S3_BUCKET = os.environ.get("PROJECT_BUCKET", "ml-project2-olist-044450716277")
S3_RAW_PREFIX = os.environ.get("RAW_PREFIX", "ml-project2-v2/data/raw/")
missing = [f for f in config.RAW_FILES.values() if not (config.RAW_DIR / f).exists()]
print("missing before download:", missing)
if missing:
    import boto3
    s3 = boto3.client("s3")
    config.RAW_DIR.mkdir(parents=True, exist_ok=True)
    for f in missing:
        s3.download_file(S3_BUCKET, S3_RAW_PREFIX + f, str(config.RAW_DIR / f))
        print("downloaded", f)
print("raw dir:", config.RAW_DIR)

missing before download: []
raw dir: /home/sagemaker-user/ml-project2-v2/data/raw


In [5]:
# Cell 5 - verify the files: size and number of data rows
inv = []
for name, f in config.RAW_FILES.items():
    p = config.RAW_DIR / f
    with open(p, "rb") as fh:
        n_lines = sum(1 for _ in fh) - 1
    inv.append({"table": name, "file": f, "size_MB": round(p.stat().st_size / 1e6, 2), "rows": n_lines})
inv = pd.DataFrame(inv); display(inv)
expected_rows = {"orders": 99441, "items": 112650, "customers": 99441, "sellers": 3095, "products": 32951,
                 "payments": 103886, "geolocation": 1000163, "translation": 71}
for t_, n_ in expected_rows.items():
    assert int(inv.loc[inv.table == t_, "rows"].iloc[0]) == n_, f"{t_}: unexpected row count"
print("Row counts match the published Olist dataset (reviews has embedded newlines, so its line count is larger than its record count).")
inv.to_csv(config.TABLE_DIR / "00_raw_file_inventory.csv", index=False)

,table,file,size_MB,rows
0,orders,olist_orders_dataset.csv,17.65,99441
1,items,olist_order_items_dataset.csv,15.44,112650
2,customers,olist_customers_dataset.csv,9.03,99441
3,sellers,olist_sellers_dataset.csv,0.17,3095
4,products,olist_products_dataset.csv,2.38,32951
5,payments,olist_order_payments_dataset.csv,5.78,103886
6,reviews,olist_order_reviews_dataset.csv,14.45,104719
7,geolocation,olist_geolocation_dataset.csv,61.27,1000163
8,translation,product_category_name_translation.csv,0.00,71


Row counts match the published Olist dataset (reviews has embedded newlines, so its line count is larger than its record count).


In [6]:
# Cell 6 - key configuration used by all notebooks
cfg = {k: getattr(config, k) for k in ["SEED", "TRAIN_START", "VAL_START", "TEST_START", "TEST_END",
                                        "LABEL_MATURITY_DAYS", "DAILY_CAPACITY", "BUDGETS", "SMOOTH_M", "OPEN_CAP_DAYS"]}
for k, v in cfg.items():
    print(f"{k:20s} = {v}")

SEED                 = 42
TRAIN_START          = 2017-01-01
VAL_START            = 2018-03-01
TEST_START           = 2018-06-01
TEST_END             = 2018-09-01
LABEL_MATURITY_DAYS  = 60
DAILY_CAPACITY       = 10
BUDGETS              = [5, 10, 20, 50]
SMOOTH_M             = 20
OPEN_CAP_DAYS        = 60
